In [5]:
!pip install optuna

In [6]:
from IPython.display import display

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim

import optuna

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCH = 10

df = pd.read_csv('./dataset/data_cleaning.csv')
X, y = df.drop(columns='cardio', inplace=False), df['cardio']

display(X.head())

,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,ag_stage,bmi,age_years,age_groups
0,2,168,62.0,110,80,1,1,0,0,1,-1,22.0,50,2
1,1,156,85.0,140,90,3,1,0,0,1,1,35.0,55,2
2,1,165,64.0,130,70,3,1,0,0,0,-1,24.0,51,2
3,2,169,82.0,150,100,1,1,0,0,1,-1,29.0,48,1
4,1,156,56.0,100,60,1,1,0,0,0,-1,23.0,47,1


In [7]:
scaler = StandardScaler()
scaled_X = scaler.fit_transform(X)

scaled_X

array([[ 1.36403989,  0.45217522, -0.8537876 , ..., -1.04528311,
        -0.4201051 ,  0.66376649],
       [-0.73311639, -1.05776727,  0.75816977, ...,  1.43570517,
         0.31880809,  0.66376649],
       [-0.73311639,  0.0746896 , -0.71361739, ..., -0.66359261,
        -0.27232246,  0.66376649],
       ...,
       [ 1.36403989,  2.33960333,  2.15987184, ...,  0.67232416,
        -0.12453983,  0.66376649],
       [-0.73311639, -0.17696748, -0.15293657, ..., -0.09105685,
         1.20550393,  0.66376649],
       [-0.73311639,  0.7038323 , -0.15293657, ..., -0.47274735,
         0.46659073,  0.66376649]], shape=(69527, 14))

**Критерий Кайзера** (правило «собственное значение > 1») — это статистический метод в факторном анализе и методе главных компонент (PCA), используемый для определения оптимального количества сохраняемых факторов. Согласно критерию, следует оставлять только те факторы (компоненты), чьи собственные значения (eigenvalues) больше 1.

In [8]:
pca = PCA()
X_pca = pca.fit(scaled_X)

variance = X_pca.explained_variance_
cnt_component = variance[variance > 1].shape[0]

display(variance)
display(cnt_component)

array([2.91060763, 2.02914845, 1.61717642, 1.45517394, 1.28506544,
       1.08098511, 0.98791338, 0.65229114, 0.53982489, 0.52771527,
       0.47638302, 0.25450954, 0.17677353, 0.0066336 ])

6

In [9]:
pca = PCA(n_components=cnt_component)
X_pca = pca.fit_transform(scaled_X)

X_pca.shape, scaled_X.shape

((69527, 6), (69527, 14))

In [10]:
tensor_X = torch.from_numpy(X_pca).to(DEVICE)
tensor_y = torch.from_numpy(y.values).to(DEVICE)


print(DEVICE)

cpu


In [11]:
class MyData(Dataset):
    def __init__(self):
        self.x = tensor_X
        self.y = tensor_y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]


data = MyData()
train_data, val_data, test_data = random_split(data, [.6, .2, .2])

train_loader = DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_data, shuffle=False, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, shuffle=False, batch_size=BATCH_SIZE)

In [55]:
def fit(model, optimizer, loss_func):
    model.train()
    for epoch in range(EPOCH):
        for data in train_loader:
            inputs, labels = data
            
            predict = model(inputs.float()).squeeze()
            loss = loss_func(predict, labels.float())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
    model.eval()



In [59]:
def objective(trial):
    model = nn.Sequential()
    
    model.add_module('input_layer', nn.Linear(6, 100))
    model.add_module('act1', nn.ReLU())
    
    overfitting = trial.suggest_categorical('overfitting', ['batchnorm', 'dropout', 'none'])
    n_layers = trial.suggest_int('n_layers', 1, 5)
    
    for i in range(n_layers):
        model.add_module(f'hidden_layer{i+2}', nn.Linear(100, 100))
        if overfitting == 'batchnorm':
            model.add_module(f'batchnorm{i}', nn.BatchNorm1d(100))
        elif overfitting == 'dropout':
            model.add_module(f'dropout{i}', nn.Dropout(0.2))
        model.add_module(f'act{i+2}', nn.ReLU())

    model.add_module('output_layer', nn.Linear(100, 1))
    model.add_module('sigmoid', nn.Sigmoid())

    optimizer = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rms'])
    learning_rate = trial.suggest_float('lr', 1e-5, 1e-1)
    if optimizer == 'adam':
        optimization_func = optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer == 'sgd':
        optimization_func = optim.SGD(model.parameters(), lr=learning_rate)
    else:
        optimization_func = optim.RMSprop(model.parameters(), lr=learning_rate)

    loss_func = nn.BCELoss()
    fit(model, optimization_func, loss_func)

    loss = 0
    for i, data in enumerate(val_loader, 1):
        inputs, labels = data
        predict = model(inputs.float()).squeeze()
        loss += loss_func(predict, labels.float())
        
    return loss / i

    
        
study = optuna.create_study(study_name='my_study', direction='minimize')
study.optimize(objective, n_trials=100)

[I 2026-03-10 18:08:11,465] A new study created in memory with name: my_study
[I 2026-03-10 18:08:43,958] Trial 0 finished with value: 0.5483750700950623 and parameters: {'overfitting': 'dropout', 'n_layers': 2, 'optimizer': 'sgd', 'lr': 0.04162001234625104}. Best is trial 0 with value: 0.5483750700950623.
[I 2026-03-10 18:09:28,489] Trial 1 finished with value: 0.5504701733589172 and parameters: {'overfitting': 'batchnorm', 'n_layers': 1, 'optimizer': 'rms', 'lr': 0.018286954459518313}. Best is trial 0 with value: 0.5483750700950623.
[I 2026-03-10 18:11:31,683] Trial 2 finished with value: 50.35623550415039 and parameters: {'overfitting': 'dropout', 'n_layers': 4, 'optimizer': 'adam', 'lr': 0.07465267825475301}. Best is trial 0 with value: 0.5483750700950623.
[I 2026-03-10 18:13:37,397] Trial 3 finished with value: 0.6227067112922668 and parameters: {'overfitting': 'dropout', 'n_layers': 4, 'optimizer': 'adam', 'lr': 0.026149880367439603}. Best is trial 0 with value: 0.548375070095062

In [60]:
study.best_params

{'overfitting': 'none',
 'n_layers': 2,
 'optimizer': 'sgd',
 'lr': 0.04350231588900495}

In [61]:
'''{'overfitting': 'none',
 'n_layers': 2,
 'optimizer': 'sgd',
 'lr': 0.04350231588900495}'''

"{'overfitting': 'none',\n 'n_layers': 2,\n 'optimizer': 'sgd',\n 'lr': 0.04350231588900495}"